# 102 Spark optimizations

The goal of this lab is to understand some of the optimization mechanisms of Spark.

- Scala
    - [Spark programming guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html)
    - [RDD APIs](https://spark.apache.org/docs/latest/api/scala/org/apache/spark/rdd/RDD.html)
    - [PairRDD APIs](https://spark.apache.org/docs/latest/api/scala/org/apache/spark/rdd/PairRDDFunctions.html)
- Python
    - [Spark programming guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html)
    - [All RDD APIs](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.html)

Use `Tab` for autocompletion, `Shift+Tab` for documentation.

In [1]:
from pyspark.sql import SparkSession

# .master("local[N]") <-- ask for N cores on the driver
spark = SparkSession.builder.master("local[4]").appName("Local Spark").config('spark.ui.port', '4040').getOrCreate()
sc = spark.sparkContext

sc

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/08 15:44:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=local[4] appName=Local Spark>

## The weather dataset

Download the following ZIP files and unzip them inside the "datasets/big" folder of this repo (which is not committed).
- [weather-sample1.txt](https://big.csr.unibo.it/downloads/bigdata/weather-datasets-s1.zip) <-- start from this!
- [weather-sample10.txt](https://big.csr.unibo.it/downloads/bigdata/weather-datasets-s10.zip)
- [weather-full.txt](https://big.csr.unibo.it/downloads/bigdata/weather-datasets-full.zip)
  
The weather datasets are textual files with weather data from all over the world in year 2000 (collected from the [National Climatic Data Center](ftp://ftp.ncdc.noaa.gov/pub/data/noaa/) of the USA. The full one weighs 13GB, the other are samples of 10% (1.3GB) and 1% (130MB) respectively.
  - Sample row: 005733213099999**19580101**03004+51317+028783FM-12+017199999V0203201N00721004501CN0100001N9 **-0021**1-01391102681
  - The date in YYYYMMDD format is located at 0-based position 15-23
  - The temperatue in x10 Celsius degrees is located at 0-based positions 87-92

In the dataset folder you also have *weather-stations.csv*; it is a structured file with the description of weather stations collecting the weather data.

In [2]:
# WEATHER structure: (usaf,wban,year,month,day,airTemperature,airTemperatureQuality)
def parseWeather(row):
    usaf = row[4:10]
    wban = row[10:15]
    year = row[15:19]
    month = row[19:21]
    day = row[21:23]
    airTemperature = row[87:92]
    airTemperatureQuality = row[92]

    return (usaf,wban,year,month,day,int(airTemperature)/10,airTemperatureQuality == '1')

# STATION structure: (usaf,wban,city,country,state,latitude,longitude,elevation,date_begin,date_end) 
def parseStation(row):
    def getDouble(str):
        return 0 if len(str)==0 else float(str)
    
    columns = [ x.replace("\"","") for x in row.split(",") ]
    latitude = getDouble(columns[6])
    longitude = getDouble(columns[7])
    elevation = getDouble(columns[8])
    return (columns[0],columns[1],columns[2],columns[3],columns[4],latitude,longitude,elevation,columns[9],columns[10])  

In [3]:
rddWeather = sc.\
  textFile("../../../../datasets/big/weather-sample10.txt").\
  map(lambda x: parseWeather(x))
rddStation = sc.\
  textFile("../../../../datasets/weather-stations.csv").\
  map(lambda x: parseStation(x))

In [8]:
import os
print(os.getcwd())

/home/blagojasavevski/Documents/lab-cse-26-27/src/main/python/lab102


## 102-1 Simple job optimization

Optimize the two jobs (avg temperature and max temperature) by avoiding the repetition of the same computations and by enforcing a partitioning criteria.
- There are multiple methods to repartition an RDD: check the ```coalesce```, ```partitionBy```, and ```repartition``` methods on the documentation and choose the best one.
- Verify your persisted data in the web UI
- Verify the execution plan of your RDDs with ```rdd.toDebugString``` or on the web UI

In [30]:
rddCached = rddWeather.\
    filter(lambda x: x[5]<999).\
    map(lambda x: (x[3], (x[5],1))).\
    coalesce(8).\
    cache()


![Cached RDD](../../../../CachedRDD.png)

In [31]:
import os
print(os.getcwd())


/home/blagojasavevski/Documents/lab-cse-26-27/src/main/python/lab102


In [32]:
# Average temperature for every month
avgTempMonth = rddCached.\
  reduceByKey(lambda v1, v2: (v1[0]+v2[0], v1[1]+v2[1])).\
  mapValues(lambda v: round(v[0]/v[1],2)).\
  collect()

In [33]:
# Maximum temperature for every month
maxTempMonth = rddCached.\
  reduceByKey(lambda x, y: y if x<y else x).\
  collect()

## 102-2 RDD preparation

Check the five possibilities to prepare the Station RDD for subsequent processing and identify the best one.

In [4]:
num_partitions = 8

In [35]:
rddS1 = rddStation.\
  keyBy(lambda x: x[0] + x[1]).\
  partitionBy(num_partitions).\
  cache().\
  map(lambda kv: (kv[0],(kv[1][3],kv[1][7])))

In [36]:
rddS2 = rddStation.\
  keyBy(lambda x: x[0] + x[1]).\
  map(lambda kv: (kv[0],(kv[1][3],kv[1][7]))).\
  cache().\
  partitionBy(num_partitions)

In [37]:
rddS3 = rddStation.\
  keyBy(lambda x: x[0] + x[1]).\
  partitionBy(num_partitions).\
  map(lambda kv: (kv[0],(kv[1][3],kv[1][7]))).\
  cache()

In [38]:
rddS4 = rddStation.\
  keyBy(lambda x: x[0] + x[1]).\
  map(lambda kv: (kv[0],(kv[1][3],kv[1][7]))).\
  partitionBy(num_partitions).\
  cache()

In [39]:

rddS5 = rddStation.\
  map(lambda x: (x[0] + x[1], (x[3],x[7]))).\
  partitionBy(num_partitions).\
  cache()

In [40]:
print(rddS5.partitioner)

## 102-3 Joining RDDs

Define the join between rddWeather and rddStation and compute:
- The maximum temperature for every city
- The maximum temperature for every city in the UK: 
  - ```StationData.country == "UK"```
- Sort the results by descending temperature
  - ```map(lambda kv: (kv[1],kv[0]))``` to invert key with value and vice versa

Hints & considerations:
- Keep only temperature values <999
- Join syntax: ```rdd1.join(rdd2)```
  - Both RDDs should be structured as key-value RDDs with the same key: usaf + wban
- Consider partitioning and caching to optimize the join
  - [Scala only] Careful: it is not enough for the two RDDs to have the same number of partitions; they must have the same partitioner! To create a partitioning function, you must ```import org.apache.spark.HashPartitioner``` and then define ```p = new HashPartitioner(n)``` where ```n``` is the number of partitions to create.
- Verify the execution plan of the join in the web UI

In [5]:
# WEATHER structure: (usaf,wban,year,month,day,airTemperature,airTemperatureQuality)
def parseWeather(row):
    usaf = row[4:10]
    wban = row[10:15]
    year = row[15:19]
    month = row[19:21]
    day = row[21:23]
    airTemperature = row[87:92]
    airTemperatureQuality = row[92]

    return (usaf,wban,year,month,day,int(airTemperature)/10,airTemperatureQuality == '1')

# STATION structure: (usaf,wban,city,country,state,latitude,longitude,elevation,date_begin,date_end) 
def parseStation(row):
    def getDouble(str):
        return 0 if len(str)==0 else float(str)
    
    columns = [ x.replace("\"","") for x in row.split(",") ]
    latitude = getDouble(columns[6])
    longitude = getDouble(columns[7])
    elevation = getDouble(columns[8])
    return (columns[0],columns[1],columns[2],columns[3],columns[4],latitude,longitude,elevation,columns[9],columns[10])  

In [41]:
rddWeather = sc.\
  textFile("../../../../datasets/big/weather-sample10.txt").\
  map(lambda x: parseWeather(x))
rddStation = sc.\
  textFile("../../../../datasets/weather-stations.csv").\
  map(lambda x: parseStation(x))

In [24]:
maxTempPerKey = rddWeather .\
filter(lambda x: x[5] < 999).\
map(lambda x: (x[0] + x[1],x[5])).\
reduceByKey(lambda x,y: max(x,y)).\
partitionBy(num_partitions).\
cache()
maxTempPerKey.take(10)


[('02670099999', 24.5),
 ('04351099999', 6.7),
 ('07630399999', 35.0),
 ('10033099999', 30.2),
 ('13577099999', 30.4),
 ('14111099999', 29.6),
 ('15237099999', 7.0),
 ('16337099999', 32.0),
 ('17609099999', 36.0),
 ('28552099999', 31.9)]

In [25]:
CityPerKey = rddStation .\
map(lambda x: (x[0] + x[1],(x[2],x[3]))).\
partitionBy(num_partitions).\
cache()
CityPerKey.take(10)


[('00702699999', ('WXPOD 7026', 'AF')),
 ('00705999999', ('CWOS 07059', '')),
 ('00706499999', ('CWOS 07064', '')),
 ('00841599999', ('XM21', '')),
 ('01001399999', ('ROST', 'NO')),
 ('01001699999', ('RORVIK/RYUM', 'NO')),
 ('01010099999', ('ANDOYA', 'NO')),
 ('01016099999', ('KONGSOYA', 'NO')),
 ('01030099999', ('KISTEFJELL', 'NO')),
 ('01030399999', ('TROMSO/SKATTURA', 'NO'))]

In [31]:
WCagg = maxTempPerKey.\
join(CityPerKey).cache()
maxTempPerCity = WCagg.\
map(lambda j:(j[1][1][0],j[1][0])) 
# (id,(temp, (city, country))) -> (city, temp)
maxTempPerCity.take(10)

[('KALMAR', 24.5),
 ('APUTITEEQ', 6.7),
 ('MONTAUBAN', 35.0),
 ('MEIERWIK', 30.2),
 ('LAZAROPOLE', 30.4),
 ('ILIRSKA BISTRICA', 29.6),
 ('BOGUS ROMANIAN', 7.0),
 ('BONIFATI (CAPE)', 32.0),
 ('LARNACA', 36.0),
 ('SADRINSK', 31.9)]

In [35]:
maxTempCityUk = WCagg.\
map(lambda j: (j[1][1][0],(j[1][0],j[1][1][1]))).\
filter(lambda j: j[1][1] == 'UK').\
map(lambda j: (j[0],j[1][0]))
maxTempCityUk.take(10)

#(id, (temp,(city,country))) -> (city, (temp, country))

[('COSFORD', 30.0),
 ('CARTERHOUSE', 22.7),
 ('LYNEHAM', 28.2),
 ('DUNKESWELL AERODROME', 22.4),
 ('BOURNEMOUTH', 25.8),
 ('ENVIRONM BUOY 63103', 14.9),
 ('NORTH RONA ISLAND', 15.1),
 ('FIFE NESS', 19.6),
 ('K1 BUOY', 18.9),
 ('LYBSTER', 18.0)]

## 102-4 Memory occupation

Use Spark's web UI to verify the space occupied by the provided RDDs.

*Warning*: in PySpark, StoraleLevels use serialization by default (see [documentation](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.StorageLevel.html)).

In [36]:
from pyspark import StorageLevel

# Clear the cache
for (id, rdd) in sc._jsc.getPersistentRDDs().items():         
    rdd.unpersist()

memSerRdd = rddWeather.cache()
memRdd = memSerRdd.map(lambda x: x).persist(StorageLevel. MEMORY_AND_DISK_DESER)
diskRdd = memSerRdd.map(lambda x: x).persist(StorageLevel.DISK_ONLY)

## 102-5 Evaluating different join methods

Consider the following scenario:
- We have a disposable RDD of Weather data (i.e., it is used only once): ```rddW```
- And we have an RDD of Station data that is used many times: ```rddS```
- Both RDDs are cached (```collect()```is called to enforce caching)

We want to join the two RDDS. Which option is best?
- Simply join the two RDDs
- Enforce on ```rddW1``` the same partitioner of ```rddS``` (and then join)
- Exploit broadcast variables

In [37]:
num_partitions = 8

rddW = rddWeather.\
  filter(lambda w: w[5]<999).\
  keyBy(lambda w: w[0]+w[1]).\
  cache()

rddS = rddStation.\
  keyBy(lambda s: s[0]+s[1]).\
  partitionBy(num_partitions).\
  cache()

# Collect to enforce caching
rddW.collect()
rddS.collect()

[('00702699999',
  ('007026',
   '99999',
   'WXPOD 7026',
   'AF',
   '',
   0.0,
   0.0,
   7026.0,
   '20120713',
   '20170715')),
 ('00705999999',
  ('007059', '99999', 'CWOS 07059', '', '', 0, 0, 0, '20120314', '20120828')),
 ('00706499999',
  ('007064', '99999', 'CWOS 07064', '', '', 0, 0, 0, '20121218', '20121219')),
 ('00841599999',
  ('008415', '99999', 'XM21', '', '', 0, 0, 0, '20131002', '20160217')),
 ('01001399999',
  ('010013', '99999', 'ROST', 'NO', '', 0, 0, 0, '19861120', '19880105')),
 ('01001699999',
  ('010016',
   '99999',
   'RORVIK/RYUM',
   'NO',
   '',
   64.85,
   11.233,
   14.0,
   '19870116',
   '19910806')),
 ('01010099999',
  ('010100',
   '99999',
   'ANDOYA',
   'NO',
   '',
   69.293,
   16.144,
   13.1,
   '19310103',
   '20170715')),
 ('01016099999',
  ('010160',
   '99999',
   'KONGSOYA',
   'NO',
   '',
   78.933,
   28.9,
   20.0,
   '19930501',
   '20170715')),
 ('01030099999',
  ('010300',
   '99999',
   'KISTEFJELL',
   'NO',
   '',
   69.283,


In [38]:
# Is it better to simply join the two RDDs..
rddX = rddW.\
  join(rddS).\
  map(lambda kv: (kv[1][1][2],kv[1][0][5])).\
  reduceByKey(lambda x,y: min(x,y),1)
print(rddX.toDebugString().decode("unicode_escape"))

(1) PythonRDD[185] at RDD at PythonRDD.scala:59 []
 |  MapPartitionsRDD[184] at mapPartitions at PythonRDD.scala:197 []
 |  ShuffledRDD[183] at partitionBy at DirectMethodHandleAccessor.java:103 []
 +-(50) PairwiseRDD[182] at reduceByKey at /tmp/ipykernel_6431/683189098.py:5 []
    |   PythonRDD[181] at reduceByKey at /tmp/ipykernel_6431/683189098.py:5 []
    |   MapPartitionsRDD[180] at mapPartitions at PythonRDD.scala:197 []
    |   ShuffledRDD[179] at partitionBy at DirectMethodHandleAccessor.java:103 []
    +-(50) PairwiseRDD[178] at join at /tmp/ipykernel_6431/683189098.py:3 []
       |   PythonRDD[177] at join at /tmp/ipykernel_6431/683189098.py:3 []
       |   UnionRDD[176] at union at DirectMethodHandleAccessor.java:103 []
       |   PythonRDD[174] at RDD at PythonRDD.scala:59 []
       |   PythonRDD[169] at RDD at PythonRDD.scala:59 []
       |       CachedPartitions: 42; MemorySize: 32.5 MiB; DiskSize: 0.0 B
       |   PythonRDD[166] at RDD at PythonRDD.scala:59 []
       |  

In [39]:
rddX.collect()

[('BANAK', -24.3),
 ('MAKKAUR FYR', -11.0),
 ('VIGRA', -3.3),
 ('FOKSTUA II', -16.2),
 ('HALLANDS VADERO', -2.0),
 ('ABOYNE NO2', -22.5),
 ('BASTREET', 0.5),
 ('BERGSTADIR', -11.6),
 ('THULE AIR BASE', -36.0),
 ('F3 - NETHERLANDS', 0.7),
 ('DEELEN', -4.1),
 ('LE GOEREE', 0.2),
 ('SAENTIS', -18.6),
 ('PORT EN BESSIN', -4.2),
 ('BRIGNOGAN', 0.3),
 ('VANNES/SENE', -2.5),
 ('LA ROCHELLE', -1.3),
 ('MONTEVENTOSO', 4.4),
 ('VIGO', 2.8),
 ('TANCOS', -0.4),
 ('HANAU AAF', -8.0),
 ('TULLN', -13.4),
 ('SONNBLICK', -26.2),
 ('PRAHA-LIBUS', -13.8),
 ('HURBANOVO', -8.6),
 ('PAKS', -12.6),
 ('LAZAROPOLE', -13.2),
 ('OSIJEK\\SPA', -8.0),
 ('MAKARSKA', 0.4),
 ('DUBROVNIK', -4.0),
 ('LIVNO', -12.2),
 ('CLUJ NAPOCA', -20.0),
 ('BOGUS ROMANIAN', -23.4),
 ('RESITA', -14.0),
 ('FAUREI/=694514 KQYB', -13.4),
 ('PLATFORMA GLORIA', -6.0),
 ('TREVISO', -7.0),
 ('TRIPOLIS', -9.8),
 ('ZONGULDAK', -2.0),
 ('KIRSEHIR', -14.0),
 ('MILAS BODRUM', -2.0),
 ('FINIKE', 3.0),
 ('TAMBEY', -36.3),
 ('IM E K FEDOROVA', -26.

In [40]:
# ..to enforce on rddW1 the same partitioner of rddS..
rddX = rddW.\
  partitionBy(num_partitions).\
  join(rddS).\
  map(lambda kv: (kv[1][1][2],kv[1][0][5])).\
  reduceByKey(lambda x,y: min(x,y),1)
print(rddX.toDebugString().decode("unicode_escape"))

(1) PythonRDD[197] at RDD at PythonRDD.scala:59 []
 |  MapPartitionsRDD[196] at mapPartitions at PythonRDD.scala:197 []
 |  ShuffledRDD[195] at partitionBy at DirectMethodHandleAccessor.java:103 []
 +-(8) PairwiseRDD[194] at reduceByKey at /tmp/ipykernel_6431/958411475.py:6 []
    |  PythonRDD[193] at reduceByKey at /tmp/ipykernel_6431/958411475.py:6 []
    |  PartitionerAwareUnionRDD[192] at union at DirectMethodHandleAccessor.java:103 []
    |  PythonRDD[190] at RDD at PythonRDD.scala:59 []
    |  MapPartitionsRDD[189] at mapPartitions at PythonRDD.scala:197 []
    |  ShuffledRDD[188] at partitionBy at DirectMethodHandleAccessor.java:103 []
    +-(42) PairwiseRDD[187] at partitionBy at /tmp/ipykernel_6431/958411475.py:3 []
       |   PythonRDD[186] at partitionBy at /tmp/ipykernel_6431/958411475.py:3 []
       |   PythonRDD[169] at RDD at PythonRDD.scala:59 []
       |       CachedPartitions: 42; MemorySize: 32.5 MiB; DiskSize: 0.0 B
       |   PythonRDD[166] at RDD at PythonRDD.scal

In [41]:
rddX.collect()

[('ANDOYA', -10.0),
 ('BODO', -11.0),
 ('VIGRA', -3.3),
 ('FOKSTUA II', -16.2),
 ('NAMDALSEID-VENGSTAD', -10.0),
 ('RORVIK/RYUM', -12.0),
 ('MERAKER-UTSYN', -12.8),
 ('FORDE-TEFRE', -6.8),
 ('MODALEN II', -14.0),
 ('LEIRIN', -28.0),
 ('ARJEPLOG', -30.0),
 ('BLOMSKOG', -12.4),
 ('MORA', -1.0),
 ('EGGEGRUND', -9.4),
 ('FILM', -16.2),
 ('NORDKOSTER', -8.6),
 ('HALLANDS VADERO', -2.0),
 ('KALMAR', -12.1),
 ('ENONTEKIO KILPISJARVI', -30.9),
 ('IVALO', -33.0),
 ('KEMI TORNIO', -26.0),
 ('OULU', -23.0),
 ('MUSTASAARI VALASSAARET', -11.0),
 ('MIKKELI', -22.0),
 ('SAVONLINNA', -27.0),
 ('KOTKA RANKKI', -13.2),
 ('ISOSAARI', -13.6),
 ('SCATSTA', -6.0),
 ('MUCKLE HOLM', -1.8),
 ('NORTH RONA ISLAND', -1.5),
 ('STORNOWAY', -2.5),
 ('LOCH GLASCARNOCH', -7.0),
 ('AULTBEA NO2', -2.7),
 ('WICK', -5.6),
 ('LYBSTER', -3.8),
 ('ABOYNE NO2', -22.5),
 ('OBAN', -0.1),
 ('FIFE NESS', -0.1),
 ('CARTERHOUSE', -7.0),
 ('ISLE OF MAN', 0.0),
 ('GREAT DUN FELL NO2', -6.7),
 ('BRIDLINGTON MRSC', -3.1),
 ('WAINFLEET 

In [ ]:
# ..or to exploit broadcast variables?
bRddS = sc.broadcast(rddS.map(lambda s: (s[0], s[1][2])).collectAsMap())
rddJ = rddW.\
  map(lambda kv: (bRddS.value.get(kv[0]), kv[1][5])).\
  filter(lambda x: x[0] is not None)

rddX = rddJ.\
  reduceByKey(lambda x,y: min(x,y),1)
print(rddX.toDebugString().decode("unicode_escape"))

In [ ]:
rddX.collect()